# Grid occupation
***
<small>Created: 14/04/2026   &emsp;   Updated : 12/05/2026<small>

La idea de este notebook es emplear el dataset contenido en ./Data/join/total_filt.nc y crear un grid de resolución modificable que contenga información sobre el número de los perfil que está contenido en cada punto del grid y el número de perfiles en el mismo. La idea es seguir la línea de grid_2025, solo que no se calcularan temperaturas medias. El dataset resultante tendrá dimensiones de $(n_{lat}, n_{lon},n_{prof})$, donde son los índices de la latitud, longitud y el número de perfiles máximo que puede tener cada punto respectivamete. Como coordenadas se tendrá:

* $time \in \mathbb{R}^{n_{lat}\times n_{lon}\times n_{n_{{prof}}}} $
* $latitude \in \mathbb{R}^{n_{lat}}$
* $longitude \in \mathbb{R}^{n_{lon}}$

Además como variables tendremos:
* $profiles \in \mathbb{R}^{n_{lat}\times n_{lon}\times n_{n_{{prof}}}}$
* $n \in  \mathbb{R}^{n_{lat}\times n_{lon}}$
* $mask \in \mathbb{R}^{n_{lat}\times n_{lon}}$
* $basin \in \mathbb{R}^{n_{lat}\times n_{lon}}$
* $batimetry \in \mathbb{R}^{n_{lat}\times n_{lon}} [m]$
* $Surface \in \mathbb{R}^{n_{lat}\times n_{lon}} [m^2]$

La batimetría se extrae del archivo con los datos de batimetría disponibles en ./Data/Batimetria/GlobalTopo2min_19.1.nc. Esta se extrae para cada pixel mediante interpolación. La superficie de cada pixel se calcula empleando la fórmula: $A = R^2 d\lambda (\sin(\phi + \frac{d\phi}{2}) - \sin(\phi -\frac{d\phi}{2})) $. Por último, la máscara y el nombre de cada cuenca se extrae de ./Data/Mascara/mascara.nc, donde el archivo es obtenido del código CreateMask.ipynb.

In [ ]:
# File manipulation packages
import os

# Data manipulation packages
import numpy as np
import datetime as datetime
import pandas as pd
import xarray as xr

# Packages for representation
import matplotlib.pyplot as plt
import cartopy
import cartopy.feature as cfeature
import cartopy.crs as ccrs

# Function for locate nearest pixel
from Locate import locate

### Apertura y muestra de archivos

In [28]:
# Open the join dataset and batimetry
path = './Data/join/total_filt.nc'
ds = xr.open_dataset(path)
batimetry = xr.open_dataset('./Data/Batimetria/GlobalTopo2min_19.1.nc')
ds

<xarray.Dataset> Size: 3GB
Dimensions:               (N_PROF: 21592, P: 6501)
Coordinates:
    time                  (N_PROF) datetime64[ns] 173kB ...
    section_id            (N_PROF) <U40 3MB ...
    file_name             (N_PROF) <U29 3MB ...
    latitude              (N_PROF) float64 173kB ...
    longitude             (N_PROF) float64 173kB ...
    pressure              (P) int64 52kB ...
Dimensions without coordinates: N_PROF, P
Data variables:
    ctd_temperature_filt  (N_PROF, P) float64 1GB ...
    ctd_salinity_filt     (N_PROF, P) float64 1GB ...
    ctd_oxygen_filt       (N_PROF, P) float64 1GB ...

In [29]:
batimetry

<xarray.Dataset> Size: 830MB
Dimensions:  (lat: 9600, lon: 21600)
Coordinates:
  * lat      (lat) float64 77kB -79.99 -79.97 -79.96 ... 79.96 79.97 79.99
  * lon      (lon) float64 173kB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
Data variables:
    z        (lat, lon) float32 829MB ...
Attributes:
    Conventions:  COARDS, CF-1.5
    title:        
    history:      grdsample -R-180/180/-80/80 -I1m @GMTAPI@-000001 -Gtopo_19....
    description:  
    GMT_version:  5.4.5 [64-bit]
    node_offset:  1

### Definición de características

In [ ]:
# Defining the necessary variables
step = 0.25 # Resolution
resolution_str = str(step).split('.')[-1] # String resolution for file name
n_prof_max = int(step * 400) # Estimated n_prof max

# Define grid
longitude_space = np.arange(-180, 180 + step, step, dtype = float)
latitude_space = np.arange(-90, 90 + step, step, dtype = float)

# Extract mask values
ds_mask = xr.open_dataset('./Data/Mascara/mascara.nc')

### Función de creación del grid

In [19]:
# Function that returns the matrices of variables and coordinates
def grid_matrixs(ds, longitude, latitude, n_prof):
    # Create nan matrices
    profiles = np.full((len(latitude), len(longitude), n_prof), np.nan) # Contained profiles
    n = np.zeros((len(latitude), len(longitude))) # Number of profiles in each grid point
    time = np.full((len(latitude), len(longitude), n_prof), np.datetime64('NaT'), dtype='datetime64[ns]') # Date of every profile


    for prof in ds.N_PROF.values:
        if np.isnan(ds.latitude.values[prof]) == True or np.isnan(ds.longitude.values[prof]) == True: continue # If there is no latitude or longitude values skip

        ds.longitude.values[prof] = ds.longitude.values[prof] - 360 if ds.longitude.values[prof] > 180 else ds.longitude.values[prof] # Transform longitude from 0-360 to -180-180

        ilat = locate(ds.latitude.values[prof], latitude)
        ilon = locate(ds.longitude.values[prof], longitude)

        for k in range(n_prof):
            if np.isnan(profiles[ilat, ilon, k]) == True:
                profiles[ilat, ilon, k] = prof
                time[ilat, ilon, k] = ds.time[prof].values
                n[ilat, ilon] += 1
                break
    
    # Cut to made n_prof = n_max
    n[np.where(n == 0)] = np.nan
    n_max = int(np.nanmax(n))
    profiles = profiles[:, :, :n_max]
    time = time[:, :, :n_max]
    n_prof = n_max
    
    return (n, profiles, time, n_prof)

In [31]:
# Execute the function
n, profiles, time, n_prof = grid_matrixs(ds, longitude_space, latitude_space, n_prof = n_prof_max)

### Batimetria

In [32]:
# Interpolate batimetry matrix
batimetry = batimetry.rename({'lat': 'latitude', 'lon' : 'longitude'})
batimetry = batimetry.interp(latitude = latitude_space, longitude = longitude_space)

### Creación del dataset

In [33]:
# Create the Dataset
profiles_ds = xr.Dataset(
    data_vars = dict(
        profiles = (["latitude", "longitude", "n_prof"], profiles)
    ),
    coords = dict(
        latitude = (["latitude"], latitude_space),
        longitude = (["longitude"], longitude_space),
        n_prof = (["n_prof"], np.arange(0, n_prof)),
        times = (["latitude", "longitude", "n_prof"], time),
        n = (["latitude", "longitude"], n),
        batimetry = (['latitude', 'longitude'], batimetry.z.values)
    )
)



### Adición de la máscara y nombre de cuenca

In [34]:
# Forcing mask and basin to have the same coordinates as ds_profiles
ds_mask_align = ds_mask.reindex_like(profiles_ds, method = 'nearest')

# Creating the variable
profiles_ds['mask'] = ds_mask_align['mask']
profiles_ds['basin'] = ds_mask_align['basin']

# Set it as a coordinate
profiles_ds = profiles_ds.set_coords(['mask'])
profiles_ds = profiles_ds.set_coords(['basin'])

### Adición de la superficie por pixel

In [ ]:
# Radius of the Earth
R = 6371000.0

# Convert the step from degrees to radians
dlat = np.deg2rad(step)
dlon = np.deg2rad(step)

# Convert latitude to radians
lat_rad = np.deg2rad(profiles_ds.latitude)

area_lat = (R**2) * dlon * (np.sin(lat_rad + dlat/2)- np.sin(lat_rad - dlat/2))

# Correct the surface on the poles
area_lat.loc[dict(latitude=90.0)] = (R**2) * dlon * (np.sin(np.deg2rad(90)) - np.sin(np.deg2rad(90 - step/2)))
area_lat.loc[dict(latitude=-90.0)] = (R**2) * dlon * (np.sin(np.deg2rad(-90 + step/2)) - np.sin(np.deg2rad(-90)))

# As soon as the surface only depends on lat, we extend
area_2d , _ = xr.broadcast(area_lat, profiles_ds.longitude)

# Load the new variable
profiles_ds['surface'] = area_2d
profiles_ds = profiles_ds.set_coords(['surface'])
profiles_ds['surface'].attrs['units'] = 'm^2'
profiles_ds['surface'].attrs['long_name'] = 'Grid cell area'

### Guardado del dataset

In [36]:
# Saving in a netcdf file
profiles_ds.to_netcdf(f"./Data/grid/occupation_grid_{resolution_str}.nc")

### Muestra

In [37]:
profiles_ds

<xarray.Dataset> Size: 1GB
Dimensions:    (latitude: 721, longitude: 1441, n_prof: 78)
Coordinates:
  * latitude   (latitude) float64 6kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0
  * longitude  (longitude) float64 12kB -180.0 -179.8 -179.5 ... 179.8 180.0
    n          (latitude, longitude) float64 8MB nan nan nan nan ... nan nan nan
    batimetry  (latitude, longitude) float64 8MB nan nan nan nan ... nan nan nan
    mask       (latitude, longitude) float64 8MB ...
    basin      (latitude, longitude) <U18 75MB ...
    surface    (latitude, longitude) float64 8MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
  * n_prof     (n_prof) int64 624B 0 1 2 3 4 5 6 7 8 ... 70 71 72 73 74 75 76 77
    times      (latitude, longitude, n_prof) datetime64[ns] 648MB NaT ... NaT
Data variables:
    profiles   (latitude, longitude, n_prof) float64 648MB nan nan ... nan nan

## Mapa de ocupaciones

In [ ]:
# Create the figure
fig = plt.figure(figsize = (20, 10))

# Add features
ax = plt.axes(projection = ccrs.PlateCarree(central_longitude=0))
ax.set_extent((-180, 180, -90, 90))
ax.coastlines()
ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.BORDERS, linestyle = ":")
bathym = cfeature.NaturalEarthFeature(name='bathymetry_F_5000', scale='10m', category='physical')
ax.add_feature(bathym, facecolor='none', edgecolor='gray', linestyle='dashed', linewidth=1, alpha = 0.2)
ax.gridlines(draw_labels=True, linewidth=1, color='gray', alpha=0.2, linestyle='--')

# Plot profiles
profiles = profiles_ds.n.plot(ax = ax, transform = ccrs.PlateCarree(), add_colorbar = False, cmap = 'gist_ncar')
fig.colorbar(mappable = profiles, label = "Number of profiles")

# Other caractheristics
ax.set_title("Occupation grid", fontsize = 20)
fig.tight_layout()

# Saving the figure
plt.savefig(f"./plots/Occupation_grids/occupation_grid_{resolution_str}.png")

### Cerrar todo

In [2]:
ds.close()
profiles_ds.close()
plt.close("all")

NameError: name 'ds' is not defined